In [0]:
import glob
import itertools
import json
import os
import sys
import warnings
from datetime import date, datetime, timedelta

import pandas as pd
import pyspark.sql.functions as F
from dateutil.relativedelta import relativedelta
from pyspark import SparkConf, SparkContext
from pyspark.sql import Column, DataFrame, SparkSession
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [0]:
ist = 3
mem = 1
core = 3


In [0]:
name = "crb_wly"


In [0]:
basic_configs = {
    "spark.app.name": name,
    "spark.master": "yarn",
    "spark.yarn.queue": "mhkd",
    "spark.memory.fraction": "0.8",
    "spark.executor.memory": f"{mem}g",
    "spark.executor.cores": core,
    "spark.sql.parquet.compression.codec": "zstd",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.sql.shuffle.partitions": "500",
    "spark.rdd.compress": "true",
    "spark.sql.legacy.timeParserPolicy": "LEGACY",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
}

schedule_configs = {
    "spark.dynamicAllocation.maxExecutors": ist,
    "spark.dynamicAllocation.initialExecutors": ist,
    "spark.network.timeout": "100s",
    "spark.executor.heartbeatInterval": "10s",
    "spark.dynamicAllocation.executorIdleTimeout": "1200s",
    "spark.dynamicAllocation.cachedExecutorIdleTimeout": "1800s",
}

s3_configs = {
    "spark.hadoop.fs.s3a.path.style.access": "true",
    "spark.hadoop.fs.s3a.committer.magic.enabled": "true",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.hadoop.fs.s3a.aws.credentials.provider": "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
}

spark_configs = {**basic_configs, **schedule_configs, **s3_configs}

spark_conf = SparkConf().setAll(spark_configs.items())

# spark_context = SparkContext(serializer=MarshalSerializer(), conf=spark_conf)
# Create SparkSession using the configured SparkConf
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()


In [0]:
spark


In [0]:
file_path = os.path.expanduser("~/env/aws_keys.json")

with open(file_path) as f:
    bucket_keys = json.load(f)

hconf = spark._jsc.hadoopConfiguration()  # type: ignore

for bucket, creds in bucket_keys.items():
    hconf.set(f"fs.s3a.bucket.{bucket}.access.key", creds["access_key"])
    hconf.set(f"fs.s3a.bucket.{bucket}.secret.key", creds["secret_key"])
    hconf.set(f"fs.s3a.bucket.{bucket}.endpoint", creds["endpoint"])

print("Configured buckets:", ", ".join(bucket_keys.keys()))


In [0]:
from itertools import product

DIM_VALUE_DICT = {
    "DAILY_FREQ": [
        "early_morning",
        "morning",
        "noon",
        "afternoon",
        "evening",
        "night",
    ],
    "MONTHLY_FREQ": ["early_month", "mid_month", "end_month"],
}


def cast_decimal_to_double(df: DataFrame) -> DataFrame:
    df_cols = df.columns
    decimal_col = [
        col for col in df_cols if isinstance(df.schema[col].dataType, DecimalType)
    ]

    for col in decimal_col:
        df = df.withColumn(col, F.col(col).cast(DoubleType()))

    return df


def select_l1w(df: DataFrame) -> DataFrame:
    res = (
        df.where("customer_code != -1")
        .groupby("customer_code", "txn_date")
        .agg(
            F.sum("vnd_balance_amount").alias("vnd_balance_amount"),
        )
        .groupby("customer_code")
        .agg(
            F.sum("vnd_balance_amount").alias("amt_sum"),
            F.min("vnd_balance_amount").alias("amt_min"),
            F.max("vnd_balance_amount").alias("amt_max"),
            F.count("txn_date").alias("day_cnt"),
        )
    )

    return res


def select_lxw(df: DataFrame) -> DataFrame:
    res = df.groupby("customer_code").agg(
        F.sum("amt_sum").alias("amt_sum"),
        F.min("amt_min").alias("amt_min"),
        F.max("amt_max").alias("amt_max"),
        F.sum("day_cnt").alias("day_cnt"),
    )

    return res


def feature_lxw(
    df: DataFrame,
    timely: str,
    dim_cols=[],
) -> DataFrame:
    prefix = "crb_casa"
    combinations = product(*(DIM_VALUE_DICT[dim.upper()] for dim in dim_cols))
    pivot_val = ["_".join([prefix] + list(comb)).lower() for comb in combinations]

    result = (
        df.withColumn("amt_avg", F.col("amt_sum") / F.col("day_cnt"))
        .withColumn("pivot_col", F.concat_ws("_", F.lit("crb_casa"), *dim_cols))
        .groupby("customer_code")
        .pivot("pivot_col", pivot_val)
        .agg(
            F.first("amt_avg").alias(f"amt_avg_{timely}"),
            F.first("amt_min").alias(f"amt_min_{timely}"),
            F.first("amt_max").alias(f"amt_max_{timely}"),
        )
    )

    result = cast_decimal_to_double(result)

    return result


# CRRB CASA

In [0]:
CASA_DIM = ["current", "guarantee", "saving", "specific"]

crb_casa_filter = F.col("group_2_dim").isin(CASA_DIM) & F.col("group_1_dim").eqNullSafe(
    "asset"
)


In [0]:
def main(layer, timely, txn_date, dims_str="base"):
    if layer == "select":
        if timely == "l1w":
            start_date = txn_date - relativedelta(weeks=1)
            end_date = txn_date - relativedelta(days=1)

            select_daily_df = (
                spark.read.csv(
                    "s3a://ttmhkd-experiment/working_zone/hainv5/aws_poc/coderpush/crb_sample_4m_20250701",
                    header=True,
                    inferSchema=True,
                )
                .where(F.col("txn_date").between(start_date, end_date))
                .where(crb_casa_filter)
            )
            slt_df = select_l1w(select_daily_df)
            slt_df.repartition(5).write.mode("overwrite").parquet(
                f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/l1w/source=crb/acc_type=casa/{dims_str}/txn_date={txn_date}"
            )
        elif timely == "l4w":
            start_date = txn_date - relativedelta(weeks=3)
            end_date = txn_date
            select_daily_df = spark.read.parquet(
                f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/l1w/source=crb/acc_type=casa/{dims_str}"
            ).where(F.col("txn_date").between(start_date, end_date))
            slt_df = select_lxw(select_daily_df)
            slt_df.repartition(5).write.mode("overwrite").parquet(
                f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/{timely}/source=crb/acc_type=casa/{dims_str}/txn_date={txn_date}"
            )
        elif timely == "l12w":
            start_date = txn_date - relativedelta(weeks=8)
            end_date = txn_date
            select_daily_df = spark.read.parquet(
                f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/l4w/source=crb/acc_type=casa/{dims_str}"
            ).where(
                F.col("txn_date").isin([
                    start_date,
                    start_date + relativedelta(weeks=4),
                    end_date,
                ])
            )
            slt_df = select_lxw(select_daily_df)
            slt_df.repartition(3).write.mode("overwrite").parquet(
                f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/{timely}/source=crb/acc_type=casa/{dims_str}/txn_date={txn_date}"
            )
        else:
            raise ValueError("Timely not support")
    elif layer == "feature":
        select_daily_df = spark.read.parquet(
            f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/select_zone/{timely}/source=crb/acc_type=casa/{dims_str}/txn_date={txn_date}"
        )
        slt_df = feature_lxw(select_daily_df, timely, [])
        slt_df.repartition(5).write.mode("overwrite").parquet(
            f"s3a://ttmhkd-experiment/working_zone/hainv5/aws/test_pipeline/feature_zone/{timely}/source=crb/acc_type=casa/{dims_str}/txn_date={txn_date}"
        )
    else:
        raise ValueError("layer not support")


In [0]:
def generate_weekly_dates(start_date: str, end_date: str):
    """Generate weekly dates (every 7 days) between start_date and end_date."""
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    dates = []
    current = start
    while current <= end:
        dates.append(current.date())
        current += relativedelta(weeks=1)
    return dates


In [0]:
layers = ["select", "feature"]
timely_list = ["l1w", "l4w", "l12w"]

dates = generate_weekly_dates("2025-03-03", "2025-07-01")
tasks = list(itertools.product(layers, timely_list, dates))


In [0]:
for params in tasks:
    print(params)
    main(*params)


In [0]:
# spark.stop()
